In [5]:
import ee, json
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
info = json.loads(key)
creds = ee.ServiceAccountCredentials(info["client_email"], key_data=key)
ee.Initialize(creds, project="lulc-sar-project")
print("Earth Engine ready:", info["client_email"])

Earth Engine ready: kaggle-ee@lulc-sar-project.iam.gserviceaccount.com


In [6]:
import glob, time, numpy as np, pandas as pd

pf = glob.glob("/kaggle/input/**/places_v2.csv", recursive=True)[0]
old = pd.read_csv(pf)
print("existing places:", len(old))

wc = ee.ImageCollection("ESA/WorldCover/v200").first()
built = wc.eq(50).rename("built")

lats = np.arange(-55, 72, 1.0)
lons = np.arange(-180, 180, 1.0)
pts = [(float(la), float(lo)) for la in lats for lo in lons]
print("grid points:", len(pts))

existing places: 1000
grid points: 45720


In [7]:
def measure(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = built.reduceRegion(ee.Reducer.mean(), box, scale=200, maxPixels=1e8).get("built")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    rows = []
    for o in out:
        s = o["properties"].get("share")
        lon, lat = o["geometry"]["coordinates"]
        if s is not None:
            rows.append((lat, lon, s))
    return rows

# pick 1000 points spread over the whole grid (every 46th point), not just the first rows
test_pts = pts[::46][:1000]
rows, step, t0 = [], 250, time.time()
for i in range(0, len(test_pts), step):
    rows += measure(test_pts[i:i+step])
    print("chunk", i // step + 1, "done, elapsed", round(time.time() - t0), "s, rows", len(rows))
print("total seconds for", len(test_pts), "points:", round(time.time() - t0))
print("share >= 0.25 in this sample:", int(sum(r[2] >= 0.25 for r in rows)))

chunk 1 done, elapsed 1 s, rows 26
chunk 2 done, elapsed 2 s, rows 96
chunk 3 done, elapsed 4 s, rows 191
chunk 4 done, elapsed 5 s, rows 349
total seconds for 994 points: 5
share >= 0.25 in this sample: 2


In [8]:
rows, step, t0 = [], 250, time.time()
for i in range(0, len(pts), step):
    try:
        rows += measure(pts[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 20 == 0:
        print(i, "/", len(pts), "elapsed", round(time.time() - t0), "s, rows", len(rows))

cand = pd.DataFrame(rows, columns=["lat", "lon", "share"])
cand.to_csv("/kaggle/working/city_grid.csv", index=False)
print("measured:", len(cand))
for t in (0.05, 0.10, 0.15, 0.25, 0.40):
    print("tiles with share >=", t, ":", int((cand.share >= t).sum()))

0 / 45720 elapsed 1 s, rows 11
5000 / 45720 elapsed 22 s, rows 200
10000 / 45720 elapsed 46 s, rows 908
15000 / 45720 elapsed 70 s, rows 2272
20000 / 45720 elapsed 92 s, rows 3676
25000 / 45720 elapsed 116 s, rows 5103
30000 / 45720 elapsed 141 s, rows 7043
35000 / 45720 elapsed 159 s, rows 9463
40000 / 45720 elapsed 182 s, rows 12480
45000 / 45720 elapsed 204 s, rows 16152
measured: 16467
tiles with share >= 0.05 : 412
tiles with share >= 0.1 : 207
tiles with share >= 0.15 : 120
tiles with share >= 0.25 : 56
tiles with share >= 0.4 : 22


In [1]:
from math import radians, sin, cos, asin, sqrt

def hav(a, b, c, d):
    p = radians
    x = sin(p(c - a) / 2) ** 2 + cos(p(a)) * cos(p(c)) * sin(p(d - b) / 2) ** 2
    return 12742 * asin(sqrt(x))

def continent(lat, lon):
    if -170 <= lon <= -30 and lat >= 12: return "North America"
    if -92 <= lon <= -30 and lat < 12: return "South America"
    if -25 <= lon <= 45 and 35 <= lat <= 72: return "Europe"
    if -20 <= lon <= 55 and -36 <= lat < 35: return "Africa"
    if 110 <= lon <= 180 and lat < -10: return "Oceania"
    if 25 <= lon <= 180 and lat >= -10: return "Asia"
    return "Other"

c = cand[cand.share >= 0.15].copy()
c["continent"] = [continent(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.continent != "Other"].sort_values("share", ascending=False).reset_index(drop=True)
oldp = list(zip(old.lat, old.lon))

def free(la, lo, chosen):
    if any(hav(la, lo, a, b) < 25 for a, b in oldp): return False
    return not any(hav(la, lo, a, b) < 25 for a, b, *_ in chosen)

chosen = []
for cont in ["North America", "South America", "Europe", "Africa", "Asia", "Oceania"]:
    n = 0
    for _, r in c[c.continent == cont].iterrows():
        if n >= 3: break
        if free(r.lat, r.lon, chosen):
            chosen.append((r.lat, r.lon, r.share, cont)); n += 1
for _, r in c.iterrows():
    if len(chosen) >= 40: break
    if free(r.lat, r.lon, chosen):
        chosen.append((r.lat, r.lon, r.share, r.continent))

out = pd.DataFrame(chosen, columns=["lat", "lon", "built_share", "continent"])
out.insert(0, "name", [f"city_{i+1:02d}" for i in range(len(out))])
out.to_csv("/kaggle/working/cities_candidates.csv", index=False)
print(len(out), "selected")
print(out.continent.value_counts().to_string())
print(out.round(3).to_string(index=False))

NameError: name 'cand' is not defined

In [2]:
import os
print(os.path.exists("/kaggle/working/city_grid.csv"))

False


In [2]:
import ee, json
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("EE_KEY")
info = json.loads(key)
creds = ee.ServiceAccountCredentials(info["client_email"], key_data=key)
ee.Initialize(creds, project="lulc-sar-project")
print("Earth Engine ready:", info["client_email"])

Earth Engine ready: kaggle-ee@lulc-sar-project.iam.gserviceaccount.com


In [3]:
import glob, time, numpy as np, pandas as pd

pf = glob.glob("/kaggle/input/**/places_v2.csv", recursive=True)[0]
old = pd.read_csv(pf)
print("existing places:", len(old))

wc = ee.ImageCollection("ESA/WorldCover/v200").first()
built = wc.eq(50).rename("built")

lats = np.arange(-55, 72, 1.0)
lons = np.arange(-180, 180, 1.0)
pts = [(float(la), float(lo)) for la in lats for lo in lons]
print("grid points:", len(pts))

existing places: 1000
grid points: 45720


In [4]:
import os

def measure(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = built.reduceRegion(ee.Reducer.mean(), box, scale=200, maxPixels=1e8).get("built")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    rows = []
    for o in out:
        s = o["properties"].get("share")
        lon, lat = o["geometry"]["coordinates"]
        if s is not None:
            rows.append((lat, lon, s))
    return rows

rows, step, t0 = [], 250, time.time()
for i in range(0, len(pts), step):
    try:
        rows += measure(pts[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 20 == 0:
        print(i, "/", len(pts), "elapsed", round(time.time() - t0), "s, rows", len(rows))
        pd.DataFrame(rows, columns=["lat", "lon", "share"]).to_csv("/kaggle/working/city_grid.csv", index=False)

cand = pd.DataFrame(rows, columns=["lat", "lon", "share"])
cand.to_csv("/kaggle/working/city_grid.csv", index=False)
print("measured:", len(cand))
print("file saved:", os.path.exists("/kaggle/working/city_grid.csv"))
for t in (0.05, 0.10, 0.15, 0.25, 0.40):
    print("tiles with share >=", t, ":", int((cand.share >= t).sum()))

0 / 45720 elapsed 0 s, rows 11
5000 / 45720 elapsed 11 s, rows 200
10000 / 45720 elapsed 19 s, rows 908
15000 / 45720 elapsed 28 s, rows 2272
20000 / 45720 elapsed 43 s, rows 3676
25000 / 45720 elapsed 51 s, rows 5103
30000 / 45720 elapsed 60 s, rows 7043
35000 / 45720 elapsed 68 s, rows 9463
40000 / 45720 elapsed 77 s, rows 12480
45000 / 45720 elapsed 88 s, rows 16152
measured: 16467
file saved: True
tiles with share >= 0.05 : 412
tiles with share >= 0.1 : 207
tiles with share >= 0.15 : 120
tiles with share >= 0.25 : 56
tiles with share >= 0.4 : 22


In [5]:
from math import radians, sin, cos, asin, sqrt

def hav(a, b, c, d):
    p = radians
    x = sin(p(c - a) / 2) ** 2 + cos(p(a)) * cos(p(c)) * sin(p(d - b) / 2) ** 2
    return 12742 * asin(sqrt(x))

def continent(lat, lon):
    if -170 <= lon <= -30 and lat >= 12: return "North America"
    if -92 <= lon <= -30 and lat < 12: return "South America"
    if -25 <= lon <= 45 and 35 <= lat <= 72: return "Europe"
    if -20 <= lon <= 55 and -36 <= lat < 35: return "Africa"
    if 110 <= lon <= 180 and lat < -10: return "Oceania"
    if 25 <= lon <= 180 and lat >= -10: return "Asia"
    return "Other"

c = cand[cand.share >= 0.15].copy()
c["continent"] = [continent(a, b) for a, b in zip(c.lat, c.lon)]
c = c[c.continent != "Other"].sort_values("share", ascending=False).reset_index(drop=True)
oldp = list(zip(old.lat, old.lon))

def free(la, lo, chosen):
    if any(hav(la, lo, a, b) < 25 for a, b in oldp): return False
    return not any(hav(la, lo, a, b) < 25 for a, b, *_ in chosen)

chosen = []
for cont in ["North America", "South America", "Europe", "Africa", "Asia", "Oceania"]:
    n = 0
    for _, r in c[c.continent == cont].iterrows():
        if n >= 3: break
        if free(r.lat, r.lon, chosen):
            chosen.append((r.lat, r.lon, r.share, cont)); n += 1
for _, r in c.iterrows():
    if len(chosen) >= 40: break
    if free(r.lat, r.lon, chosen):
        chosen.append((r.lat, r.lon, r.share, r.continent))

out = pd.DataFrame(chosen, columns=["lat", "lon", "built_share", "continent"])
out.insert(0, "name", [f"city_{i+1:02d}" for i in range(len(out))])
out.to_csv("/kaggle/working/cities_candidates.csv", index=False)
print(len(out), "selected")
print(out.continent.value_counts().to_string())
print(out.round(3).to_string(index=False))

40 selected
continent
Asia             16
North America    12
Europe            5
Africa            4
South America     3
   name  lat  lon  built_share     continent
city_01   43  -88        0.682 North America
city_02   51 -114        0.615 North America
city_03   34 -118        0.560 North America
city_04  -12  -77        0.548 South America
city_05  -23  -47        0.331 South America
city_06  -35  -58        0.209 South America
city_07   45   39        0.443        Europe
city_08   41   29        0.400        Europe
city_09   51    7        0.368        Europe
city_10   32   36        0.544        Africa
city_11   34   -5        0.389        Africa
city_12  -26   28        0.349        Africa
city_13   35  137        0.669          Asia
city_14   23  114        0.667          Asia
city_15   11   77        0.661          Asia
city_16   23  113        0.574          Asia
city_17   33  -97        0.546 North America
city_18   40  -83        0.537 North America
city_19   33  114      

In [6]:
import io, zipfile, urllib.request
import pandas as pd, geopandas as gpd

def get(url):
    return urllib.request.urlopen(url, timeout=120).read()

# 1) GeoNames: all cities with at least 15,000 inhabitants
z = zipfile.ZipFile(io.BytesIO(get("https://download.geonames.org/export/dump/cities15000.zip")))
cols = ["geonameid","name","asciiname","alt","lat","lon","fclass","fcode","cc","cc2",
        "a1","a2","a3","a4","pop","elev","dem","tz","mod"]
gn = pd.read_csv(z.open("cities15000.txt"), sep="\t", header=None, names=cols,
                 usecols=["geonameid","name","lat","lon","cc","pop"], keep_default_na=False)
print("cities:", len(gn), "| with population >= 100,000:", int((gn["pop"] >= 100000).sum()))

# 2) Natural Earth country polygons (with continent)
zb = get("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
open("/kaggle/working/ne_countries.zip", "wb").write(zb)
world = gpd.read_file("/kaggle/working/ne_countries.zip")
print(world.columns.tolist()[:12])
print(world["CONTINENT"].value_counts().to_string())

cities: 34197 | with population >= 100,000: 6287
['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF']
CONTINENT
Africa                     51
Asia                       47
Europe                     39
North America              18
South America              13
Oceania                     7
Seven seas (open ocean)     1
Antarctica                  1


In [7]:
from math import radians, sin, cos, asin, sqrt

def hav(a, b, c, d):
    p = radians
    x = sin(p(c - a) / 2) ** 2 + cos(p(a)) * cos(p(c)) * sin(p(d - b) / 2) ** 2
    return 12742 * asin(sqrt(x))

demo = {"Sydney": (-33.8688, 151.2093), "Singapore": (1.3521, 103.8198)}
months = [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 10)]

for name, (la, lo) in demo.items():
    d = old.apply(lambda r: hav(la, lo, r.lat, r.lon), axis=1)
    i = d.idxmin()
    print(f"\n=== {name} ===")
    print("nearest training place: id", int(old.loc[i, "id"]), "at", round(d[i], 1), "km  (need >= 25)")
    pt = ee.Geometry.Point([lo, la])
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(pt)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    counts = []
    for y, m in months:
        a = f"{y}-{m:02d}-01"
        b = f"{y+1}-01-01" if m == 12 else f"{y}-{m+1:02d}-01"
        counts.append(base.filterDate(a, b).size().getInfo())
    print("scenes per month (Jan 2025 ... Sep 2026):", counts)
    print("months with zero scenes:", sum(c == 0 for c in counts))


=== Sydney ===
nearest training place: id 29 at 0.1 km  (need >= 25)
scenes per month (Jan 2025 ... Sep 2026): [8, 7, 8, 7, 8, 7, 8, 8, 8, 7, 8, 8, 7, 7, 8, 7, 6, 6, 6, 5, 5]
months with zero scenes: 0

=== Singapore ===
nearest training place: id 34 at 329.2 km  (need >= 25)
scenes per month (Jan 2025 ... Sep 2026): [6, 4, 6, 4, 4, 2, 3, 4, 6, 5, 5, 6, 4, 5, 3, 5, 2, 6, 5, 4, 5]
months with zero scenes: 0


In [8]:
cands = {"Brisbane": (-27.4698, 153.0251), "Perth": (-31.9505, 115.8605),
         "Auckland": (-36.8485, 174.7633), "Johannesburg": (-26.2041, 28.0473),
         "Santiago": (-33.4489, -70.6693), "Lagos": (6.5244, 3.3792),
         "Jakarta": (-6.2088, 106.8456), "Nairobi": (-1.2921, 36.8219)}
for name, (la, lo) in cands.items():
    d = old.apply(lambda r: hav(la, lo, r.lat, r.lon), axis=1)
    i = d.idxmin()
    print(f"{name:13s} nearest place id {int(old.loc[i,'id']):4d} at {d[i]:7.1f} km", "OK" if d[i] >= 25 else "TOO CLOSE")

Brisbane      nearest place id   71 at   163.7 km OK
Perth         nearest place id   53 at     0.1 km TOO CLOSE
Auckland      nearest place id   30 at   845.8 km OK
Johannesburg  nearest place id  658 at   344.6 km OK
Santiago      nearest place id  611 at   146.3 km OK
Lagos         nearest place id   42 at     0.5 km TOO CLOSE
Jakarta       nearest place id   33 at     1.1 km TOO CLOSE
Nairobi       nearest place id   41 at     0.3 km TOO CLOSE


In [9]:
months = [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 10)]
for name, (la, lo) in {"Brisbane": (-27.4698, 153.0251), "Auckland": (-36.8485, 174.7633)}.items():
    pt = ee.Geometry.Point([lo, la])
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(pt)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    counts = []
    for y, m in months:
        a = f"{y}-{m:02d}-01"
        b = f"{y+1}-01-01" if m == 12 else f"{y}-{m+1:02d}-01"
        counts.append(base.filterDate(a, b).size().getInfo())
    print(name, counts, "| zero months:", sum(c == 0 for c in counts))

Brisbane [3, 2, 3, 2, 3, 2, 3, 3, 2, 3, 2, 3, 2, 3, 2, 3, 1, 0, 3, 2, 3] | zero months: 1
Auckland [7, 8, 8, 10, 10, 7, 8, 8, 8, 8, 7, 8, 7, 8, 7, 11, 9, 9, 8, 7, 8] | zero months: 0


In [10]:
from shapely.geometry import Point

# continent for every city by polygon lookup; nearest country if a point falls outside all polygons
pts_gdf = gpd.GeoDataFrame(gn.copy(), geometry=gpd.points_from_xy(gn.lon, gn.lat), crs="EPSG:4326")
world_c = world[["CONTINENT", "ADMIN", "geometry"]].to_crs("EPSG:4326")
j = gpd.sjoin(pts_gdf, world_c, how="left", predicate="within")
j = j[~j.index.duplicated(keep="first")]

miss = j["CONTINENT"].isna()
print("cities outside all polygons:", int(miss.sum()))
if miss.any():
    proj = "EPSG:3857"
    near = gpd.sjoin_nearest(pts_gdf.loc[miss.values].to_crs(proj), world_c.to_crs(proj),
                             how="left", distance_col="d")
    near = near[~near.index.duplicated(keep="first")]
    j.loc[near.index, "CONTINENT"] = near["CONTINENT"]
    j.loc[near.index, "ADMIN"] = near["ADMIN"]

cities = pd.DataFrame(j.drop(columns=["geometry", "index_right"], errors="ignore"))
cities = cities[cities.CONTINENT.isin(["Africa","Asia","Europe","North America","South America","Oceania"])]
cities = cities[(cities["pop"] >= 50000) & (cities.lat.between(-56, 72))].reset_index(drop=True)
print("candidate cities (pop >= 50,000):", len(cities))
print(cities.CONTINENT.value_counts().to_string())
cities.to_csv("/kaggle/working/cities_all.csv", index=False)
print("saved:", os.path.exists("/kaggle/working/cities_all.csv"))

cities outside all polygons: 1322
candidate cities (pop >= 50,000): 12396
CONTINENT
Asia             5615
Europe           2227
Africa           1624
North America    1579
South America    1281
Oceania            70
saved: True


In [11]:
from math import radians, sin, cos, asin, sqrt
import numpy as np

# 25 km exclusion against training places and the two demo cities
excl = list(zip(old.lat, old.lon)) + [(1.3521, 103.8198), (-36.8485, 174.7633)]
ex = np.radians(np.array(excl))

def min_dist_km(la, lo):
    p1, l1 = np.radians(la), np.radians(lo)
    a = np.sin((ex[:, 0] - p1) / 2) ** 2 + np.cos(p1) * np.cos(ex[:, 0]) * np.sin((ex[:, 1] - l1) / 2) ** 2
    return float((12742 * np.arcsin(np.sqrt(a))).min())

cities["min_km"] = [min_dist_km(a, b) for a, b in zip(cities.lat, cities.lon)]
free_c = cities[cities.min_km >= 25].reset_index(drop=True)
print("candidates at least 25 km from all training places and demo cities:", len(free_c))
print(free_c.CONTINENT.value_counts().to_string())

def measure_c(df):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {"gid": int(r.geonameid)})
                               for r in df.itertuples()])
    def f(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = built.reduceRegion(ee.Reducer.mean(), box, scale=100, maxPixels=1e8).get("built")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    return [(o["properties"]["gid"], o["properties"].get("share")) for o in out]

res, step, t0 = [], 200, time.time()
for i in range(0, len(free_c), step):
    try:
        res += measure_c(free_c.iloc[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 10 == 0:
        print(i, "/", len(free_c), "elapsed", round(time.time() - t0), "s")

sh = pd.DataFrame(res, columns=["geonameid", "share"])
free_c = free_c.merge(sh, on="geonameid", how="left")
free_c.to_csv("/kaggle/working/cities_measured.csv", index=False)
print("measured:", int(free_c.share.notna().sum()))
print(free_c.groupby("CONTINENT").share.describe()[["count", "50%", "max"]].round(3).to_string())
print("saved:", os.path.exists("/kaggle/working/cities_measured.csv"))

candidates at least 25 km from all training places and demo cities: 11731
CONTINENT
Asia             5309
Europe           2154
Africa           1556
North America    1501
South America    1149
Oceania            62
0 / 11731 elapsed 1 s
2000 / 11731 elapsed 5 s
4000 / 11731 elapsed 9 s
6000 / 11731 elapsed 13 s
8000 / 11731 elapsed 17 s
10000 / 11731 elapsed 20 s
measured: 11731
                count    50%    max
CONTINENT                          
Africa         1556.0  0.146  0.918
Asia           5309.0  0.194  0.958
Europe         2154.0  0.242  0.894
North America  1501.0  0.330  0.981
Oceania          62.0  0.234  0.744
South America  1149.0  0.173  0.919
saved: True


In [12]:
from math import radians, sin, cos, asin, sqrt

def hav(a, b, c, d):
    p = radians
    x = sin(p(c - a) / 2) ** 2 + cos(p(a)) * cos(p(c)) * sin(p(d - b) / 2) ** 2
    return 12742 * asin(sqrt(x))

quota = {"Asia": 8, "Africa": 7, "North America": 7, "Europe": 6, "South America": 6, "Oceania": 6}
floor = {"Oceania": 0.10}
bins = [("large", 1_000_000, 10**9), ("mid", 200_000, 1_000_000), ("small", 50_000, 200_000)]

def size_bin(p):
    for n, lo, hi in bins:
        if lo <= p < hi: return n

fc2 = free_c[free_c.share.notna()].copy()
fc2["size"] = fc2["pop"].map(size_bin)
chosen = []

def ok(r):
    return all(hav(r.lat, r.lon, c.lat, c.lon) >= 100 for c in chosen)

for cont, q in quota.items():
    pool = fc2[(fc2.CONTINENT == cont) & (fc2.share >= floor.get(cont, 0.20))]
    per = [q // 3 + (1 if k < q % 3 else 0) for k in range(3)]   # large, mid, small
    for (bname, _, _), n in zip(bins, per):
        got = 0
        for r in pool[pool["size"] == bname].sort_values("share", ascending=False).itertuples():
            if got >= n: break
            if ok(r):
                chosen.append(r); got += 1
    # top up inside the continent if a size group ran short
    have = sum(1 for c in chosen if c.CONTINENT == cont)
    for r in pool.sort_values("share", ascending=False).itertuples():
        if have >= q: break
        if r.geonameid not in {c.geonameid for c in chosen} and ok(r):
            chosen.append(r); have += 1

sel = pd.DataFrame([c._asdict() for c in chosen]).drop(columns="Index")
sel = sel[["geonameid", "name", "ADMIN", "CONTINENT", "lat", "lon", "pop", "size", "share", "min_km"]]
sel.insert(0, "id", range(1001, 1001 + len(sel)))
sel.to_csv("/kaggle/working/cities_selected.csv", index=False)
print(len(sel), "selected")
print(sel.CONTINENT.value_counts().to_string())
print(sel["size"].value_counts().to_string())
print(sel[["id","name","ADMIN","CONTINENT","pop","size","share"]].round(3).to_string(index=False))

40 selected
CONTINENT
Asia             8
Africa           7
North America    7
Europe           6
South America    6
Oceania          6
size
large    15
mid      13
small    12
  id                name                            ADMIN     CONTINENT      pop  size  share
1001        Shijiazhuang                            China          Asia  5758403 large  0.915
1002              Nagoya                            Japan          Asia  2332176 large  0.914
1003           Zhengzhou                            China          Asia  6650532 large  0.907
1004        Higashiosaka                            Japan          Asia   493940   mid  0.958
1005           Quận Mười                          Vietnam          Asia   399000   mid  0.886
1006           Samarkand                       Uzbekistan          Asia   595200   mid  0.874
1007            San Juan                      Philippines          Asia   134312 small  0.898
1008           Din Daeng                         Thailand          Asia

In [13]:
import os, pandas as pd, numpy as np
free_c = pd.read_csv("/kaggle/working/cities_measured.csv")
print(len(free_c), int(free_c.share.notna().sum()))

11731 11731


In [14]:
from math import radians, sin, cos, asin, sqrt

def hav(a, b, c, d):
    p = radians
    x = sin(p(c - a) / 2) ** 2 + cos(p(a)) * cos(p(c)) * sin(p(d - b) / 2) ** 2
    return 12742 * asin(sqrt(x))

d = free_c[free_c.share.notna() & (free_c["pop"] >= 60000)].copy()
d = d[d.name != "Brisbane"]
d = d.sort_values("pop", ascending=False).reset_index(drop=True)
print("before cluster rule:", len(d))

# keep a city only if no more populous kept city lies within 60 km
kept = []
for r in d.itertuples():
    if all(hav(r.lat, r.lon, k.lat, k.lon) >= 60 for k in kept):
        kept.append(r)

cl = pd.DataFrame([k._asdict() for k in kept]).drop(columns="Index")
cl["band"] = cl.share.map(lambda s: "low" if s < 0.40 else ("mid" if s < 0.65 else "high"))
cl["size"] = cl["pop"].map(lambda p: "large" if p >= 1_000_000 else ("mid" if p >= 200_000 else "small"))
cl.to_csv("/kaggle/working/cities_clustered.csv", index=False)

print("after cluster rule:", len(cl))
print(cl.CONTINENT.value_counts().to_string())
print(pd.crosstab(cl.CONTINENT, cl.band).to_string())

before cluster rule: 9848
after cluster rule: 3675
CONTINENT
Asia             1509
Africa            680
Europe            619
South America     433
North America     399
Oceania            35
band           high   low  mid
CONTINENT                     
Africa           23   582   75
Asia            128  1094  287
Europe           13   499  107
North America    36   259  104
Oceania           2    29    4
South America    19   355   59


In [15]:
quota = {"Asia": 8, "Africa": 7, "North America": 7, "Europe": 6, "South America": 6, "Oceania": 6}
floor = {"Oceania": 0.10}
chosen = []

def ok(r):
    return all(hav(r.lat, r.lon, c.lat, c.lon) >= 150 for c in chosen)

rng = np.random.default_rng(7)
for cont, q in quota.items():
    pool = cl[(cl.CONTINENT == cont) & (cl.share >= floor.get(cont, 0.20))]
    order = ["low", "mid", "high"]
    per = {b: q // 3 + (1 if k < q % 3 else 0) for k, b in enumerate(order)}
    for b in order:
        sub = pool[pool.band == b].assign(rnd=rng.random((pool.band == b).sum())).sort_values("rnd")
        got, used = 0, set()
        for sz in ["large", "mid", "small"] * 4:
            if got >= per[b]: break
            for r in sub[sub["size"] == sz].itertuples():
                if r.geonameid in used: continue
                if ok(r):
                    chosen.append(r); used.add(r.geonameid); got += 1; break
    have = sum(1 for c in chosen if c.CONTINENT == cont)
    for r in pool.sort_values("pop", ascending=False).itertuples():
        if have >= q: break
        if r.geonameid not in {c.geonameid for c in chosen} and ok(r):
            chosen.append(r); have += 1

sel = pd.DataFrame([c._asdict() for c in chosen]).drop(columns="Index")
sel = sel[["geonameid","name","ADMIN","CONTINENT","lat","lon","pop","size","band","share","min_km"]]
sel.insert(0, "id", range(1001, 1001 + len(sel)))
sel.to_csv("/kaggle/working/cities_selected.csv", index=False)
print(len(sel), "selected")
print(sel.CONTINENT.value_counts().to_string())
print(sel.band.value_counts().to_string())
print(sel["size"].value_counts().to_string())
print(sel[["id","name","ADMIN","CONTINENT","pop","size","band","share"]].round(3).to_string(index=False))

40 selected
CONTINENT
Asia             8
Africa           7
North America    7
Europe           6
South America    6
Oceania          6
band
low     16
mid     13
high    11
size
mid      18
large    16
small     6
  id                 name                    ADMIN     CONTINENT     pop  size band  share
1001                Yunfu                    China          Asia 2612800 large  low  0.221
1002               Luzhou                    China          Asia  998900   mid  low  0.396
1003         Nakhon Sawan                 Thailand          Asia   82305 small  low  0.272
1004               Huzhou                    China          Asia 1015937 large  mid  0.442
1005          Banjarmasin                Indonesia          Asia  657663   mid  mid  0.467
1006 Kattaqo’rg’on Shahri               Uzbekistan          Asia   90600 small  mid  0.417
1007                Hefei                    China          Asia 9465881 large high  0.823
1008                  Qom                     Iran       

In [16]:
months = [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 10)]
rows = []
for r in sel.itertuples():
    pt = ee.Geometry.Point([r.lon, r.lat])
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(pt)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    c = []
    for y, m in months:
        a = f"{y}-{m:02d}-01"
        b = f"{y+1}-01-01" if m == 12 else f"{y}-{m+1:02d}-01"
        c.append(base.filterDate(a, b).size().getInfo())
    rows.append((r.id, r.name, sum(x == 0 for x in c), min(c), round(sum(c) / len(c), 1)))
    print(r.id, r.name, "zero months:", rows[-1][2], "min:", rows[-1][3], "mean:", rows[-1][4])
cov = pd.DataFrame(rows, columns=["id", "name", "zero_months", "min_scenes", "mean_scenes"])
cov.to_csv("/kaggle/working/cities_coverage.csv", index=False)

1001 Yunfu zero months: 0 min: 3 mean: 4.5
1002 Luzhou zero months: 1 min: 0 mean: 1.8
1003 Nakhon Sawan zero months: 0 min: 3 mean: 5.7
1004 Huzhou zero months: 1 min: 0 mean: 2.7
1005 Banjarmasin zero months: 0 min: 2 mean: 2.5
1006 Kattaqo’rg’on Shahri zero months: 0 min: 4 mean: 4.8
1007 Hefei zero months: 0 min: 2 mean: 2.7
1008 Qom zero months: 0 min: 3 mean: 5.0
1009 Rabat zero months: 0 min: 4 mean: 10.0
1010 Tete zero months: 0 min: 2 mean: 4.7
1011 Ho zero months: 0 min: 2 mean: 5.0
1012 Alexandria zero months: 0 min: 7 mean: 14.5
1013 Minna zero months: 0 min: 1 mean: 2.9
1014 Yaoundé zero months: 0 min: 1 mean: 3.0
1015 Abeokuta zero months: 0 min: 2 mean: 3.3
1016 Jacksonville zero months: 0 min: 1 mean: 2.2
1017 Lexington zero months: 0 min: 2 mean: 4.1
1018 Bridgeport zero months: 0 min: 4 mean: 6.7
1019 Toronto zero months: 0 min: 3 mean: 4.6
1020 Sacramento zero months: 0 min: 8 mean: 16.4
1021 Montréal zero months: 0 min: 3 mean: 5.0
1022 Culiacán zero months: 0 min: 

In [17]:
repl = {}
for cont, band_ in [("Europe", "mid"), ("Asia", "low")]:
    pool = cl[(cl.CONTINENT == cont) & (cl.band == band_) & (cl.share >= 0.20)
              & (~cl.geonameid.isin(sel.geonameid)) & (cl["pop"] >= 150000)]
    keep = sel[~sel.id.isin([1026, 1002])]
    cands = []
    for r in pool.sort_values("pop", ascending=False).itertuples():
        if all(hav(r.lat, r.lon, k.lat, k.lon) >= 150 for k in keep.itertuples()):
            cands.append(r)
        if len(cands) == 6: break
    repl[cont] = cands
    print(cont, band_, [(c.name, c.ADMIN, int(c.pop), round(c.share, 2)) for c in cands])

Europe mid [('Berlin', 'Germany', 3426354, 0.64), ('Kyiv', 'Ukraine', 2952301, 0.48), ('Rome', 'Italy', 2318895, 0.63), ('Hamburg', 'Germany', 1973896, 0.56), ('Minsk', 'Belarus', 1742124, 0.63), ('Budapest', 'Hungary', 1741041, 0.49)]
Asia low [('Wuzhong', 'China', 7202654, 0.37), ('Yantai', 'China', 7102100, 0.32), ('Shantou', 'China', 3838900, 0.39), ('Puyang', 'China', 3590000, 0.27), ('Shiyan', 'China', 3460000, 0.35), ('Lüliang', 'China', 3346500, 0.21)]


In [18]:
months = [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 10)]
want = ["Budapest", "Rome", "Shantou", "Yantai"]
rc = cl[cl.name.isin(want) & cl.CONTINENT.isin(["Europe", "Asia"])]
for r in rc.itertuples():
    pt = ee.Geometry.Point([r.lon, r.lat])
    base = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(pt)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    c = []
    for y, m in months:
        a = f"{y}-{m:02d}-01"
        b = f"{y+1}-01-01" if m == 12 else f"{y}-{m+1:02d}-01"
        c.append(base.filterDate(a, b).size().getInfo())
    print(r.name, r.ADMIN, "zero months:", sum(x == 0 for x in c), "min:", min(c), "mean:", round(sum(c)/len(c), 1))

Yantai China zero months: 0 min: 3 mean: 5.2
Shantou China zero months: 0 min: 1 mean: 2.4
Rome Italy zero months: 0 min: 7 mean: 14.8
Budapest Hungary zero months: 0 min: 5 mean: 9.8


In [19]:
final = sel[~sel.id.isin([1026, 1002])].copy()
add = cl[((cl.name == "Budapest") & (cl.CONTINENT == "Europe")) | ((cl.name == "Yantai") & (cl.CONTINENT == "Asia"))]
add = add[["geonameid","name","ADMIN","CONTINENT","lat","lon","pop","size","band","share","min_km"]]
final = pd.concat([final.drop(columns="id"), add], ignore_index=True)

# Russia east of 60E counts as Asia in the continent table (Natural Earth puts all of Russia in Europe)
final["continent_fixed"] = final.apply(
    lambda r: "Asia" if (r.ADMIN == "Russia" and r.lon > 60) else r.CONTINENT, axis=1)

final.insert(0, "id", range(1001, 1001 + len(final)))
final["group"] = "city"
final.to_csv("/kaggle/working/cities_final.csv", index=False)

print(len(final), "cities")
print(final.continent_fixed.value_counts().to_string())
print(final[["id","name","ADMIN","continent_fixed","band","size","share"]].round(3).to_string(index=False))
print("saved:", os.path.exists("/kaggle/working/cities_final.csv"))

40 cities
continent_fixed
Asia             9
Africa           7
North America    7
South America    6
Oceania          6
Europe           5
  id                 name                    ADMIN continent_fixed band  size  share
1001                Yunfu                    China            Asia  low large  0.221
1002         Nakhon Sawan                 Thailand            Asia  low small  0.272
1003               Huzhou                    China            Asia  mid large  0.442
1004          Banjarmasin                Indonesia            Asia  mid   mid  0.467
1005 Kattaqo’rg’on Shahri               Uzbekistan            Asia  mid small  0.417
1006                Hefei                    China            Asia high large  0.823
1007                  Qom                     Iran            Asia high   mid  0.710
1008                Rabat                  Morocco          Africa  low large  0.368
1009                 Tete               Mozambique          Africa  low   mid  0.340
1010      

In [20]:
rome = cl[(cl.name == "Rome") & (cl.CONTINENT == "Europe")].copy()
r = rome.iloc[0]
d = min(hav(r.lat, r.lon, k.lat, k.lon) for k in final.itertuples())
print("Rome nearest selected city (km):", round(d, 1), "(need >= 150)")
if d >= 150:
    rome = rome[["geonameid","name","ADMIN","CONTINENT","lat","lon","pop","size","band","share","min_km"]]
    rome["continent_fixed"] = "Europe"; rome["id"] = 1041; rome["group"] = "city"
    final = pd.concat([final, rome], ignore_index=True)
    final.to_csv("/kaggle/working/cities_final.csv", index=False)
print(len(final), "cities;", final.continent_fixed.value_counts().to_dict())

Rome nearest selected city (km): 749.1 (need >= 150)
41 cities; {'Asia': 9, 'Africa': 7, 'North America': 7, 'Europe': 6, 'South America': 6, 'Oceania': 6}


In [21]:
print("cl" in globals(), "final" in globals(), "old" in globals())

True True True


In [22]:
import time, numpy as np, pandas as pd

dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
      .filterDate("2025-01-01", "2026-11-01")
      .select("label"))
mode = dw.mode()                      # same rule as our labels: per-pixel mode over the period
flood = mode.eq(3).rename("flood")    # class 3 = flooded vegetation

def measure_w(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(5000).bounds()
        s = flood.reduceRegion(ee.Reducer.mean(), box, scale=100, maxPixels=1e8).get("flood")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    rows = []
    for o in out:
        s = o["properties"].get("share")
        lon, lat = o["geometry"]["coordinates"]
        if s is not None:
            rows.append((lat, lon, s))
    return rows

# timing test: 250 points spread over the whole grid
test_pts = pts[::183][:250]
t0 = time.time()
r = measure_w(test_pts)
print("250 points took", round(time.time() - t0), "s; rows:", len(r),
      "; with share >= 0.10:", sum(x[2] >= 0.10 for x in r))

250 points took 21 s; rows: 128 ; with share >= 0.10: 0


In [23]:
wl = ee.ImageCollection("ESA/WorldCover/v200").first()
wet = wl.eq(90).Or(wl.eq(95)).rename("wet")

def measure_wet(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(25000).bounds()      # 50x50 km window
        s = wet.reduceRegion(ee.Reducer.mean(), box, scale=500, maxPixels=1e8).get("wet")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0], o["properties"].get("share"))
            for o in out if o["properties"].get("share") is not None]

grid = [(float(la), float(lo)) for la in np.arange(-55, 72, 1.0) for lo in np.arange(-180, 180, 1.0)]
rows, step, t0 = [], 250, time.time()
for i in range(0, len(grid), step):
    try:
        rows += measure_wet(grid[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 30 == 0:
        print(i, "/", len(grid), "elapsed", round(time.time() - t0), "s, rows", len(rows))

wg = pd.DataFrame(rows, columns=["lat", "lon", "wet"])
wg.to_csv("/kaggle/working/wetland_grid.csv", index=False)
print("saved:", os.path.exists("/kaggle/working/wetland_grid.csv"), "| measured:", len(wg))
for t in (0.02, 0.05, 0.10, 0.20):
    print("50 km windows with wetland share >=", t, ":", int((wg.wet >= t).sum()))

0 / 45720 elapsed 1 s, rows 14
7500 / 45720 elapsed 29 s, rows 470
15000 / 45720 elapsed 54 s, rows 2493
22500 / 45720 elapsed 77 s, rows 4796
30000 / 45720 elapsed 102 s, rows 7687
37500 / 45720 elapsed 128 s, rows 11671
45000 / 45720 elapsed 155 s, rows 17172
saved: True | measured: 17529
50 km windows with wetland share >= 0.02 : 1944
50 km windows with wetland share >= 0.05 : 1299
50 km windows with wetland share >= 0.1 : 897
50 km windows with wetland share >= 0.2 : 514


In [24]:
cw = wg[wg.wet >= 0.05].copy()

excl = list(zip(old.lat, old.lon)) + [(1.3521, 103.8198), (-36.8485, 174.7633)] \
       + list(zip(final.lat, final.lon))
ex = np.radians(np.array(excl))

def min_dist_km(la, lo):
    p1, l1 = np.radians(la), np.radians(lo)
    a = np.sin((ex[:, 0] - p1) / 2) ** 2 + np.cos(p1) * np.cos(ex[:, 0]) * np.sin((ex[:, 1] - l1) / 2) ** 2
    return float((12742 * np.arcsin(np.sqrt(a))).min())

cw["min_km"] = [min_dist_km(a, b) for a, b in zip(cw.lat, cw.lon)]
cw = cw[cw.min_km >= 25].reset_index(drop=True)
print("wetland windows far enough from all existing places:", len(cw))

rows, step, t0 = [], 100, time.time()
pts_w = list(zip(cw.lat, cw.lon))
for i in range(0, len(pts_w), step):
    try:
        rows += measure_w(pts_w[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 5 == 0:
        print(i, "/", len(pts_w), "elapsed", round(time.time() - t0), "s, rows", len(rows))

dwm = pd.DataFrame(rows, columns=["lat", "lon", "flood"])
cwm = cw.merge(dwm, on=["lat", "lon"], how="left")
cwm.to_csv("/kaggle/working/wetland_measured.csv", index=False)
print("saved:", os.path.exists("/kaggle/working/wetland_measured.csv"), "| measured:", int(cwm.flood.notna().sum()))
for t in (0.02, 0.05, 0.10, 0.20):
    print("tiles with Dynamic World flooded share >=", t, ":", int((cwm.flood >= t).sum()))

wetland windows far enough from all existing places: 1279
0 / 1279 elapsed 9 s, rows 100
500 / 1279 elapsed 77 s, rows 600
1000 / 1279 elapsed 204 s, rows 1100
saved: True | measured: 1279
tiles with Dynamic World flooded share >= 0.02 : 232
tiles with Dynamic World flooded share >= 0.05 : 131
tiles with Dynamic World flooded share >= 0.1 : 75
tiles with Dynamic World flooded share >= 0.2 : 25


In [25]:
top = cwm[cwm.flood >= 0.05].copy().reset_index(drop=True)
offs = [-0.1, -0.05, 0.0, 0.05, 0.1]
best = []
t0 = time.time()
for k, r in enumerate(top.itertuples()):
    cand_pts = [(r.lat + dy, r.lon + dx) for dy in offs for dx in offs]
    try:
        rr = measure_w(cand_pts)
    except Exception as e:
        rr = [(r.lat, r.lon, r.flood)]
    b = max(rr, key=lambda x: x[2])
    best.append((b[0], b[1], b[2]))
    if k % 20 == 0:
        print(k, "/", len(top), "elapsed", round(time.time() - t0), "s")

top["lat_r"] = [b[0] for b in best]
top["lon_r"] = [b[1] for b in best]
top["flood_r"] = [b[2] for b in best]
top.to_csv("/kaggle/working/wetland_refined.csv", index=False)
print("saved:", os.path.exists("/kaggle/working/wetland_refined.csv"))
print(top[["flood", "flood_r"]].describe().round(3).to_string())

0 / 131 elapsed 2 s
20 / 131 elapsed 69 s
40 / 131 elapsed 130 s
60 / 131 elapsed 213 s
80 / 131 elapsed 293 s
100 / 131 elapsed 347 s
120 / 131 elapsed 452 s
saved: True
         flood  flood_r
count  131.000  131.000
mean     0.148    0.258
std      0.115    0.163
min      0.051    0.052
25%      0.071    0.134
50%      0.109    0.217
75%      0.161    0.336
max      0.740    0.858


In [26]:
CONT = lambda la, lo: ("Oceania" if (lo >= 110 and la < -10) else
    "North America" if (-170 <= lo <= -30 and la >= 12) else
    "South America" if (-92 <= lo <= -30 and la < 12) else
    "Europe" if (-25 <= lo <= 60 and 35 <= la <= 72) else
    "Africa" if (-20 <= lo <= 55 and -36 <= la < 35) else
    "Asia" if lo >= 25 else "Other")

top["continent"] = [CONT(a, b) for a, b in zip(top.lat_r, top.lon_r)]
top["min_km_r"] = [min_dist_km(a, b) for a, b in zip(top.lat_r, top.lon_r)]
pool_all = top[(top.flood_r >= 0.10) & (top.min_km_r >= 25) & (top.continent != "Other")]
print(pool_all.continent.value_counts().to_string())

quota = {"Africa": 5, "Asia": 5, "North America": 4, "South America": 5, "Europe": 3, "Oceania": 3}
rng = np.random.default_rng(11)
chosen = []
def ok(r):
    return all(hav(r.lat_r, r.lon_r, c.lat_r, c.lon_r) >= 150 for c in chosen)

for cont, q in quota.items():
    pool = pool_all[pool_all.continent == cont].copy()
    pool["rnd"] = rng.random(len(pool))
    hi = pool[pool.flood_r >= pool.flood_r.median()].sort_values("rnd")
    lo = pool[pool.flood_r < pool.flood_r.median()].sort_values("rnd")
    order = [x for pair in zip(list(hi.itertuples()), list(lo.itertuples())) for x in pair]
    order += list(hi.itertuples())[len(lo):] + list(lo.itertuples())[len(hi):]
    n = 0
    for r in order:
        if n >= q: break
        if ok(r):
            chosen.append(r); n += 1

wsel = pd.DataFrame([c._asdict() for c in chosen]).drop(columns="Index")
wsel = wsel[["lat_r", "lon_r", "continent", "wet", "flood_r", "min_km_r"]]
wsel.columns = ["lat", "lon", "continent", "wc_wetland_share", "dw_flood_share", "min_km"]
wsel.insert(0, "id", range(1042, 1042 + len(wsel)))
wsel["group"] = "wetland"
wsel.to_csv("/kaggle/working/wetlands_selected.csv", index=False)
print(len(wsel), "selected")
print(wsel.continent.value_counts().to_string())
print(wsel.round(3).to_string(index=False))

continent
Asia             42
North America    30
Africa           23
South America    11
Europe            8
Oceania           1
23 selected
continent
Africa           5
Asia             5
South America    5
North America    4
Europe           3
Oceania          1
  id    lat     lon     continent  wc_wetland_share  dw_flood_share  min_km   group
1042 -12.05   29.95        Africa             0.568           0.421  48.866 wetland
1043  -8.95   26.05        Africa             0.163           0.213 345.084 wetland
1044   8.90  -13.10        Africa             0.195           0.643 131.222 wetland
1045 -19.00   22.90        Africa             0.530           0.234  33.358 wetland
1046 -10.90   27.05        Africa             0.100           0.414 328.648 wetland
1047  58.05   76.00          Asia             0.207           0.314  33.905 wetland
1048  60.05   84.90          Asia             0.314           0.439 156.379 wetland
1049  62.05   82.00          Asia             0.388           

In [27]:
mg = ee.ImageCollection("ESA/WorldCover/v200").first().eq(95).rename("mang")

def measure_m(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(25000).bounds()
        s = mg.reduceRegion(ee.Reducer.mean(), box, scale=500, maxPixels=1e8).get("mang")
        return feat.set("share", s)
    out = fc.map(f).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0], o["properties"].get("share"))
            for o in out if o["properties"].get("share") is not None]

# mangroves only grow in the tropics and subtropics: limit the grid to 35 S - 32 N
grid_m = [(float(la), float(lo)) for la in np.arange(-35, 33, 1.0) for lo in np.arange(-180, 180, 1.0)]
print("grid points:", len(grid_m))

rows, step, t0 = [], 250, time.time()
for i in range(0, len(grid_m), step):
    try:
        rows += measure_m(grid_m[i:i+step])
    except Exception as e:
        print("chunk", i, "failed:", str(e)[:80])
    if (i // step) % 30 == 0:
        print(i, "/", len(grid_m), "elapsed", round(time.time() - t0), "s, rows", len(rows))

mgd = pd.DataFrame(rows, columns=["lat", "lon", "mang"])
mgd.to_csv("/kaggle/working/mangrove_grid.csv", index=False)
print("saved:", os.path.exists("/kaggle/working/mangrove_grid.csv"), "| measured:", len(mgd))
for t in (0.005, 0.01, 0.02, 0.05):
    print("windows with mangrove share >=", t, ":", int((mgd.mang >= t).sum()))

grid points: 24480
0 / 24480 elapsed 1 s, rows 22
7500 / 24480 elapsed 27 s, rows 2006
15000 / 24480 elapsed 56 s, rows 4303
22500 / 24480 elapsed 81 s, rows 7123
saved: True | measured: 7975
windows with mangrove share >= 0.005 : 337
windows with mangrove share >= 0.01 : 238
windows with mangrove share >= 0.02 : 154
windows with mangrove share >= 0.05 : 68


In [28]:
ex_all = excl + list(zip(wsel.lat, wsel.lon))
ex2 = np.radians(np.array(ex_all))
def min_d2(la, lo):
    p1, l1 = np.radians(la), np.radians(lo)
    a = np.sin((ex2[:, 0] - p1) / 2) ** 2 + np.cos(p1) * np.cos(ex2[:, 0]) * np.sin((ex2[:, 1] - l1) / 2) ** 2
    return float((12742 * np.arcsin(np.sqrt(a))).min())

mc = mgd[mgd.mang >= 0.02].copy()
mc["min_km"] = [min_d2(a, b) for a, b in zip(mc.lat, mc.lon)]
mc = mc[mc.min_km >= 25].reset_index(drop=True)
print("mangrove windows usable:", len(mc))

def measure_mang10(chunk):
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la])) for la, lo in chunk])
    def f(feat):
        box = feat.geometry().buffer(5000).bounds()
        m = mg.reduceRegion(ee.Reducer.mean(), box, scale=30, maxPixels=1e8).get("mang")
        d = flood.reduceRegion(ee.Reducer.mean(), box, scale=100, maxPixels=1e8).get("flood")
        return feat.set("mang10", m).set("flood", d)
    out = fc.map(f).getInfo()["features"]
    return [(o["geometry"]["coordinates"][1], o["geometry"]["coordinates"][0],
             o["properties"].get("mang10") or 0, o["properties"].get("flood") or 0) for o in out]

offs = [-0.1, -0.05, 0.0, 0.05, 0.1]
best, t0 = [], time.time()
for k, r in enumerate(mc.itertuples()):
    cand_pts = [(r.lat + dy, r.lon + dx) for dy in offs for dx in offs]
    try:
        rr = measure_mang10(cand_pts)
        b = max(rr, key=lambda x: x[2])
    except Exception as e:
        b = (r.lat, r.lon, 0, 0)
    best.append(b)
    if k % 20 == 0:
        print(k, "/", len(mc), "elapsed", round(time.time() - t0), "s")

mc["lat_r"] = [b[0] for b in best]; mc["lon_r"] = [b[1] for b in best]
mc["mang10"] = [b[2] for b in best]; mc["flood_r"] = [b[3] for b in best]
mc["continent"] = [CONT(a, b) for a, b in zip(mc.lat_r, mc.lon_r)]
mc["min_km_r"] = [min_d2(a, b) for a, b in zip(mc.lat_r, mc.lon_r)]
mc.to_csv("/kaggle/working/mangrove_refined.csv", index=False)
print("saved refined:", len(mc))

pool = mc[(mc.mang10 >= 0.05) & (mc.min_km_r >= 25) & (mc.continent != "Other")]
print(pool.continent.value_counts().to_string())

quota = {"Asia": 3, "Africa": 2, "South America": 1, "North America": 1}
rng = np.random.default_rng(13)
msel_rows = []
allw = [(r.lat, r.lon) for r in wsel.itertuples()]
def ok_m(r):
    pts_ = allw + [(c.lat_r, c.lon_r) for c in msel_rows]
    return all(hav(r.lat_r, r.lon_r, a, b) >= 150 for a, b in pts_)
for cont, q in quota.items():
    p = pool[pool.continent == cont].copy()
    p["rnd"] = rng.random(len(p))
    n = 0
    for r in p.sort_values("rnd").itertuples():
        if n >= q: break
        if ok_m(r):
            msel_rows.append(r); n += 1

msel = pd.DataFrame([c._asdict() for c in msel_rows]).drop(columns="Index")
msel = msel[["lat_r", "lon_r", "continent", "mang10", "flood_r", "min_km_r"]]
msel.columns = ["lat", "lon", "continent", "wc_mangrove_share", "dw_flood_share", "min_km"]
msel.insert(0, "id", range(1065, 1065 + len(msel)))
msel["group"] = "mangrove"
msel.to_csv("/kaggle/working/mangroves_selected.csv", index=False)
print(len(msel), "selected")
print(msel.round(3).to_string(index=False))

mangrove windows usable: 146
0 / 146 elapsed 3 s
20 / 146 elapsed 52 s
40 / 146 elapsed 86 s
60 / 146 elapsed 117 s
80 / 146 elapsed 144 s
100 / 146 elapsed 172 s
120 / 146 elapsed 209 s
140 / 146 elapsed 269 s
saved refined: 146
continent
Asia             54
North America    21
Africa           20
South America    19
Oceania          16
7 selected
  id    lat    lon     continent  wc_mangrove_share  dw_flood_share  min_km    group
1065  -8.10 140.00          Asia              0.325           0.001 386.366 mangrove
1066  -0.10 109.10          Asia              0.101           0.001 216.975 mangrove
1067   5.95 118.05          Asia              0.306           0.004 625.082 mangrove
1068 -14.90  47.90        Africa              0.289           0.009 275.748 mangrove
1069  -2.10  41.10        Africa              0.291           0.215 309.926 mangrove
1070 -13.90 -39.05 South America              0.325           0.012 114.995 mangrove
1071  21.90 -77.90 North America              0.160   

In [1]:
print("old" in globals(), "final" in globals(), "wsel" in globals())

False False False
